# CS689 Term Project – Update 3: ETL Process
**Student:** Medha Prodduturi  
**Topic:** Audience Consumption Evolution: Traditional Media vs. Digital Streaming Platforms

---

## Scope
This notebook covers the first iteration of the ETL process for two dimensions and one fact table:

| Table | Type | Source |
|---|---|---|
| `DIM_PLATFORM` | Dimension (SCD Type 2) | `platform_summary.csv` |
| `DIM_SUBSCRIPTION_PLAN` | Dimension (SCD Type 2) | `streaming_service.csv` |
| `FACT_SUBSCRIPTION_PRICING` | Fact (Transactional) | `streaming_service.csv` + `platform_summary.csv` |

---
# Part 7 – Loading Data into Staging

In [1]:
import pandas as pd
import numpy as np

In [2]:
StreamingService_df = pd.read_csv('Data Sources copy/streaming_service.csv')
StreamingService_df.head()

,service,date,price
0,Netflix,Jul-2011,7.99
1,Netflix,Aug-2011,7.99
2,Netflix,Sep-2011,7.99
3,Netflix,Oct-2011,7.99
4,Netflix,Nov-2011,7.99


In [3]:
PlatformSummary_df = pd.read_csv('Data Sources copy/Streaming Platform Dataset/platform_summary.csv')
PlatformSummary_df.head()

,platform,launch_year,parent_company,content_focus,business_model,reporting_date,subscribers_millions,quarterly_revenue_usd_millions,quarterly_profit_usd_millions,quarterly_eps_usd,...,projected_2026_subscribers_millions,sports_rights_spend_2026_usd_millions,sports_rights_global_share_pct,streaming_revenue_q4_usd_millions,streaming_revenue_growth_pct,monthly_streams_millions,streams_growth_pct,monthly_hours_watched_millions,monthly_hours_growth_pct,peak_concurrent_viewers_millions
0,Netflix,2007,Netflix Inc.,"Original series, films, licensed content",Subscription + Ads,2026-01-19,325.0,12050.0,2410.0,0.56,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Disney+,2019,The Walt Disney Company,"Family, Marvel, Star Wars, National Geographic",Subscription + Ads,2026-02-07,126.0,NaN,NaN,NaN,...,284.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Hulu,2008,The Walt Disney Company,"TV shows, next-day broadcast",Subscription + Ads,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Amazon Prime Video,2006,Amazon.com Inc.,"Original series, movies, sports",Prime bundle + Subscription,2026-02-06,NaN,NaN,NaN,NaN,...,NaN,3800.0,27.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Paramount+,2021,Paramount Global,"Nickelodeon, CBS, Showtime",Subscription + Ads,2026-02-06,NaN,NaN,NaN,NaN,...,NaN,1136.0,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
print("StreamingService_df shape:", StreamingService_df.shape)
print("PlatformSummary_df shape:", PlatformSummary_df.shape)

StreamingService_df shape: (777, 3)
PlatformSummary_df shape: (12, 36)


In [5]:
print("Record Count of StreamingService_df:", len(StreamingService_df))
print("Record Count of PlatformSummary_df:", len(PlatformSummary_df))

Record Count of StreamingService_df: 777
Record Count of PlatformSummary_df: 12


In [6]:
# print("StreamingService_df counts:", StreamingService_df.count())
# print("\nPlatformSummary_df counts:", PlatformSummary_df.count())

---
# Part 8 – Transforming the Data

### StreamingService_df – Duplicate Check

In [7]:
StreamingService_df.value_counts()

service     date      price
Apple TV+   Apr-2020  4.99     1
Paramount+  Oct-2015  9.99     1
            Mar-2022  9.99     1
            Mar-2023  9.99     1
            May-2015  9.99     1
                              ..
Hulu        May-2023  14.99    1
            Nov-2015  11.99    1
            Nov-2016  11.99    1
            Nov-2017  11.99    1
Shudder     Sep-2023  6.99     1
Name: count, Length: 777, dtype: int64

In [8]:
print(StreamingService_df.groupby(['service', 'date', 'price']).size().reset_index(name='count'))

       service      date  price  count
0    Apple TV+  Apr-2020   4.99      1
1    Apple TV+  Apr-2021   4.99      1
2    Apple TV+  Apr-2022   4.99      1
3    Apple TV+  Apr-2023   6.99      1
4    Apple TV+  Aug-2020   4.99      1
..         ...       ...    ...    ...
772    Shudder  Sep-2019   5.99      1
773    Shudder  Sep-2020   5.99      1
774    Shudder  Sep-2021   5.99      1
775    Shudder  Sep-2022   5.99      1
776    Shudder  Sep-2023   6.99      1

[777 rows x 4 columns]


In [9]:
StreamingService_df.describe(include='all')

,service,date,price
count,777,777,777.000000
unique,10,151,NaN
top,Netflix,Jan-2024,NaN
freq,151,10,NaN
mean,NaN,NaN,9.378674
std,NaN,NaN,2.609859
min,NaN,NaN,4.990000
25%,NaN,NaN,7.990000
50%,NaN,NaN,8.990000
75%,NaN,NaN,9.990000


In [10]:
StreamingService_df.isnull().sum()

service    0
date       0
price      0
dtype: int64

In [11]:
StreamingService_df = StreamingService_df.fillna('')
StreamingService_df.isnull().sum()

service    0
date       0
price      0
dtype: int64

In [12]:
StreamingServiceDistinct_df = StreamingService_df.drop_duplicates()

In [13]:
print('Rows of original:', StreamingService_df.shape[0])
print('Rows of distinct df:', StreamingServiceDistinct_df.shape[0])
print(f'\nRecords removed: {StreamingService_df.shape[0] - StreamingServiceDistinct_df.shape[0]}')

Rows of original: 777
Rows of distinct df: 777

Records removed: 0


In [14]:
print('Rows and columns of distinct df:', StreamingServiceDistinct_df.shape)
print('Record Count of distinct df:', len(StreamingServiceDistinct_df))
print('StreamingServiceDistinct_df counts:', StreamingServiceDistinct_df.count())

Rows and columns of distinct df: (777, 3)
Record Count of distinct df: 777
StreamingServiceDistinct_df counts: service    777
date       777
price      777
dtype: int64


### Transformation 1 – Date Standardization
The `date` column uses abbreviated month-year strings (e.g. `Jul-2011`). Standardizing to `YYYY-MM-DD` (first of month assumed) so all date keys share a consistent format across fact tables.

In [15]:
print('BEFORE transformation')
print(StreamingServiceDistinct_df[['service', 'date', 'price']].head(5).to_string())

StreamingServiceDistinct_df['date'] = pd.to_datetime(
    StreamingServiceDistinct_df['date'], format='%b-%Y'
)

print('\nAFTER transformation')
print(StreamingServiceDistinct_df[['service', 'date', 'price']].head(5).to_string())

BEFORE transformation
   service      date  price
0  Netflix  Jul-2011   7.99
1  Netflix  Aug-2011   7.99
2  Netflix  Sep-2011   7.99
3  Netflix  Oct-2011   7.99
4  Netflix  Nov-2011   7.99

AFTER transformation
   service       date  price
0  Netflix 2011-07-01   7.99
1  Netflix 2011-08-01   7.99
2  Netflix 2011-09-01   7.99
3  Netflix 2011-10-01   7.99
4  Netflix 2011-11-01   7.99


### Transformation 2 – Year Extraction (BQ #2: When did the shift accelerate?)
Extracting `year` from the standardized date to support time-series analysis of when pricing trends accelerated — a direct input to answering when the shift from traditional to digital media occurred.

In [16]:
StreamingServiceDistinct_df['year'] = StreamingServiceDistinct_df['date'].dt.year

print('AFTER transformation – year extracted')
print(StreamingServiceDistinct_df[['service', 'date', 'year', 'price']].head(10).to_string())

AFTER transformation – year extracted
   service       date  year  price
0  Netflix 2011-07-01  2011   7.99
1  Netflix 2011-08-01  2011   7.99
2  Netflix 2011-09-01  2011   7.99
3  Netflix 2011-10-01  2011   7.99
4  Netflix 2011-11-01  2011   7.99
5  Netflix 2011-12-01  2011   7.99
6  Netflix 2012-01-01  2012   7.99
7  Netflix 2012-02-01  2012   7.99
8  Netflix 2012-03-01  2012   7.99
9  Netflix 2012-04-01  2012   7.99


### Transformation 3 – Price Tier Derivation (BQ #4: What factors influence users to switch?)
Binning `price` into categorical tiers. Grouping platforms by price range lets us compare whether budget vs. premium pricing correlates with subscriber growth or churn — a key factor in BQ #4.

In [17]:
print('BEFORE transformation')
print(StreamingServiceDistinct_df[['service', 'price']].describe())

bins   = [0, 7, 14, float('inf')]
labels = ['Budget ($0-$7)', 'Mid ($7-$14)', 'Premium (>$14)']
StreamingServiceDistinct_df['price_tier'] = pd.cut(
    StreamingServiceDistinct_df['price'], bins=bins, labels=labels, right=True
)

print('\nAFTER transformation')
print(StreamingServiceDistinct_df[['service', 'date', 'price', 'price_tier']].head(10).to_string())

BEFORE transformation
            price
count  777.000000
mean     9.378674
std      2.609859
min      4.990000
25%      7.990000
50%      8.990000
75%      9.990000
max     17.990000

AFTER transformation
   service       date  price    price_tier
0  Netflix 2011-07-01   7.99  Mid ($7-$14)
1  Netflix 2011-08-01   7.99  Mid ($7-$14)
2  Netflix 2011-09-01   7.99  Mid ($7-$14)
3  Netflix 2011-10-01   7.99  Mid ($7-$14)
4  Netflix 2011-11-01   7.99  Mid ($7-$14)
5  Netflix 2011-12-01   7.99  Mid ($7-$14)
6  Netflix 2012-01-01   7.99  Mid ($7-$14)
7  Netflix 2012-02-01   7.99  Mid ($7-$14)
8  Netflix 2012-03-01   7.99  Mid ($7-$14)
9  Netflix 2012-04-01   7.99  Mid ($7-$14)


In [18]:
print('Tier distribution')
print(StreamingServiceDistinct_df['price_tier'].value_counts())

Tier distribution
price_tier
Mid ($7-$14)      560
Budget ($0-$7)    154
Premium (>$14)     63
Name: count, dtype: int64


### Transformation 4 – Month-over-Month Price Change (BQ #4: What factors influence users to switch?)
Calculating how much the price changed each month per service. Price increases are one of the most cited factors in subscriber churn — this measure directly supports answering BQ #4.

In [19]:
StreamingServiceDistinct_df = StreamingServiceDistinct_df.sort_values(['service', 'date'])

StreamingServiceDistinct_df['price_change_mom'] = (
    StreamingServiceDistinct_df.groupby('service')['price'].diff()
)

print('Months where price changed:')
price_changes = StreamingServiceDistinct_df[StreamingServiceDistinct_df['price_change_mom'] != 0].dropna(subset=['price_change_mom'])
print(price_changes[['service', 'date', 'price', 'price_change_mom']].to_string())

Months where price changed:
         service       date  price  price_change_mom
588    Apple TV+ 2022-10-01   6.99               2.0
603    Apple TV+ 2024-01-01   9.99               3.0
167      Disney+ 2021-03-01   7.99               1.0
188      Disney+ 2022-12-01  10.99               3.0
198      Disney+ 2023-10-01  13.99               3.0
235      HBO Max 2023-02-01  15.99               1.0
319         Hulu 2021-10-01  12.99               1.0
333         Hulu 2022-12-01  14.99               2.0
346         Hulu 2024-01-01  17.99               3.0
90       Netflix 2019-01-01   8.99               1.0
126      Netflix 2022-01-01   9.99               1.0
147      Netflix 2023-10-01  15.49               5.5
451   Paramount+ 2023-06-01  11.99               2.0
641      Peacock 2023-08-01  11.99               2.0
552  Prime Video 2024-01-01  11.99               3.0
729      Shudder 2023-08-01   6.99               1.0


### Transformation 4 – Cumulative Price Increase Since Launch (BQ #1: Which platforms show growth/decline?)
Measuring how much each platform's price has risen since its first recorded month. Platforms that raised prices aggressively can be compared against engagement trends from `FACT_PLATFORM_ENGAGEMENT` to answer BQ #1.

In [20]:
StreamingServiceDistinct_df['cumulative_price_increase'] = (
    StreamingServiceDistinct_df.groupby('service')['price']
    .transform(lambda x: x - x.iloc[0])
)

print('Cumulative price increase per service (latest month):')
latest = (
    StreamingServiceDistinct_df.sort_values('date')
    .groupby('service')
    .last()
    .reset_index()
)
print(latest[['service', 'date', 'price', 'cumulative_price_increase']].to_string())

Cumulative price increase per service (latest month):
       service       date  price  cumulative_price_increase
0    Apple TV+ 2024-01-01   9.99                        5.0
1  Crunchyroll 2024-01-01   7.99                        0.0
2      Disney+ 2024-01-01  13.99                        7.0
3      HBO Max 2024-01-01  15.99                        1.0
4         Hulu 2024-01-01  17.99                        6.0
5      Netflix 2024-01-01  15.49                        7.5
6   Paramount+ 2024-01-01  11.99                        2.0
7      Peacock 2024-01-01  11.99                        2.0
8  Prime Video 2024-01-01  11.99                        3.0
9      Shudder 2024-01-01   6.99                        1.0


---
### PlatformSummary_df – Duplicate Check

In [21]:
PlatformSummary_df['platform'].value_counts()

platform
Netflix               1
Disney+               1
Hulu                  1
Amazon Prime Video    1
Paramount+            1
Peacock               1
Apple TV+             1
Max                   1
Roku                  1
YouTube TV            1
Tubi                  1
Pluto TV              1
Name: count, dtype: int64

In [22]:
dim_cols = ['platform', 'parent_company', 'content_focus', 'business_model', 'launch_year']
print(PlatformSummary_df.groupby(dim_cols).size().reset_index(name='count'))

              platform           parent_company  \
0   Amazon Prime Video          Amazon.com Inc.   
1            Apple TV+               Apple Inc.   
2              Disney+  The Walt Disney Company   
3                 Hulu  The Walt Disney Company   
4                  Max   Warner Bros. Discovery   
5              Netflix             Netflix Inc.   
6           Paramount+         Paramount Global   
7              Peacock                  Comcast   
8             Pluto TV         Paramount Global   
9                 Roku                Roku Inc.   
10                Tubi          Fox Corporation   
11          YouTube TV                   Google   

                                     content_focus  \
0                  Original series, movies, sports   
1                           Original series, films   
2   Family, Marvel, Star Wars, National Geographic   
3                     TV shows, next-day broadcast   
4                     HBO, Discovery, Warner Bros.   
5         Or

In [23]:
dim_cols = ['platform', 'parent_company', 'content_focus', 'business_model', 'launch_year']
PlatformSummary_df[dim_cols].describe(include='all')

,platform,parent_company,content_focus,business_model,launch_year
count,12,12,12,12,12.000000
unique,12,10,11,6,NaN
top,Netflix,The Walt Disney Company,"FAST, library content",Subscription + Ads,NaN
freq,1,2,2,6,NaN
mean,NaN,NaN,NaN,NaN,2014.416667
std,NaN,NaN,NaN,NaN,5.743903
min,NaN,NaN,NaN,NaN,2006.000000
25%,NaN,NaN,NaN,NaN,2008.000000
50%,NaN,NaN,NaN,NaN,2015.500000
75%,NaN,NaN,NaN,NaN,2019.250000


In [24]:
PlatformSummary_df[dim_cols].isnull().sum()

platform          0
parent_company    0
content_focus     0
business_model    0
launch_year       0
dtype: int64

In [25]:
PlatformSummary_df = PlatformSummary_df.fillna('')
PlatformSummary_df[dim_cols].isnull().sum()

platform          0
parent_company    0
content_focus     0
business_model    0
launch_year       0
dtype: int64

In [26]:
PlatformSummaryDistinct_df = PlatformSummary_df[dim_cols].drop_duplicates()

In [27]:
print('Rows of original:', PlatformSummary_df.shape[0])
print('Rows of distinct df:', PlatformSummaryDistinct_df.shape[0])
print(f'\nRecords removed: {PlatformSummary_df.shape[0] - PlatformSummaryDistinct_df.shape[0]}')

Rows of original: 12
Rows of distinct df: 12

Records removed: 0


In [28]:
print('Rows and columns of distinct df:', PlatformSummaryDistinct_df.shape)
print('Record Count of distinct df:', len(PlatformSummaryDistinct_df))
print('PlatformSummaryDistinct_df counts:', PlatformSummaryDistinct_df.count())

Rows and columns of distinct df: (12, 5)
Record Count of distinct df: 12
PlatformSummaryDistinct_df counts: platform          12
parent_company    12
content_focus     12
business_model    12
launch_year       12
dtype: int64


### Transformation 6 – Derived Columns for DIM_PLATFORM (BQ #1 & BQ #2)
`is_digital` and `media_sector` enable direct comparison between digital and traditional platforms (BQ #1). `launch_decade` groups platforms by the era they entered the market, supporting BQ #2 analysis of when the digital shift accelerated.

In [29]:
print('BEFORE transformation')
print(PlatformSummaryDistinct_df.to_string())

# All platforms in this source are digital streaming services
PlatformSummaryDistinct_df = PlatformSummaryDistinct_df.copy()
PlatformSummaryDistinct_df['is_digital']          = True
PlatformSummaryDistinct_df['media_sector']        = 'Streaming'
PlatformSummaryDistinct_df['platform_age_years']  = 2026 - PlatformSummaryDistinct_df['launch_year'].astype(int)
PlatformSummaryDistinct_df['launch_decade']       = (PlatformSummaryDistinct_df['launch_year'].astype(int) // 10 * 10).astype(str) + 's'

print('\nAFTER transformation')
print(PlatformSummaryDistinct_df[['platform', 'launch_year', 'launch_decade', 'is_digital', 'media_sector', 'platform_age_years']].to_string())

BEFORE transformation
              platform           parent_company                                   content_focus               business_model  launch_year
0              Netflix             Netflix Inc.        Original series, films, licensed content           Subscription + Ads         2007
1              Disney+  The Walt Disney Company  Family, Marvel, Star Wars, National Geographic           Subscription + Ads         2019
2                 Hulu  The Walt Disney Company                    TV shows, next-day broadcast           Subscription + Ads         2008
3   Amazon Prime Video          Amazon.com Inc.                 Original series, movies, sports  Prime bundle + Subscription         2006
4           Paramount+         Paramount Global                      Nickelodeon, CBS, Showtime           Subscription + Ads         2021
5              Peacock                  Comcast                            NBCUniversal content           Subscription + Ads         2020
6           

### Transformation 7 – Service Name Mapping & Adding Unmatched Platforms
Normalizing service names across both sources so the fact table can join to DIM_PLATFORM. Crunchyroll and Shudder appear in pricing data but not in platform_summary — following the same approach as Assignment 3, these are identified and added to the dimension.

In [30]:
services_in_pricing  = set(StreamingServiceDistinct_df['service'].unique())
platforms_in_summary = set(PlatformSummaryDistinct_df['platform'].unique())

print('In pricing but NOT in platform_summary (need mapping or add):')
print(sorted(services_in_pricing - platforms_in_summary))

print('\nIn platform_summary but NOT in pricing (no pricing history):')
print(sorted(platforms_in_summary - services_in_pricing))

In pricing but NOT in platform_summary (need mapping or add):
['Crunchyroll', 'HBO Max', 'Prime Video', 'Shudder']

In platform_summary but NOT in pricing (no pricing history):
['Amazon Prime Video', 'Max', 'Pluto TV', 'Roku', 'Tubi', 'YouTube TV']


In [31]:
name_map = {
    'HBO Max'     : 'Max',
    'Prime Video' : 'Amazon Prime Video'
}
StreamingServiceDistinct_df['platform_name'] = StreamingServiceDistinct_df['service'].replace(name_map)

print('Service → platform_name mapping applied:')
print(StreamingServiceDistinct_df[['service', 'platform_name']].drop_duplicates().to_string())

Service → platform_name mapping applied:
         service       platform_name
553    Apple TV+           Apple TV+
735  Crunchyroll         Crunchyroll
151      Disney+             Disney+
202      HBO Max                 Max
247         Hulu                Hulu
0        Netflix             Netflix
347   Paramount+          Paramount+
604      Peacock             Peacock
459  Prime Video  Amazon Prime Video
647      Shudder             Shudder


---
# Part 9 – Loading Data into Dimensions and Fact Table

### DIM_PLATFORM (SCD Type 2)
Starting with the 12 platforms from `PlatformSummaryDistinct_df`, we merge the pricing transaction source to find any platform names not yet in the dimension — the same merge pattern used in Assignment 3 to find new ships.

In [32]:
PlatformMerge_df = pd.merge(
    StreamingServiceDistinct_df[['platform_name']].drop_duplicates(),
    PlatformSummaryDistinct_df,
    left_on='platform_name',
    right_on='platform',
    how='left',
    indicator=True
)
print(PlatformMerge_df)

        platform_name            platform           parent_company  \
0           Apple TV+           Apple TV+               Apple Inc.   
1         Crunchyroll                 NaN                      NaN   
2             Disney+             Disney+  The Walt Disney Company   
3                 Max                 Max   Warner Bros. Discovery   
4                Hulu                Hulu  The Walt Disney Company   
5             Netflix             Netflix             Netflix Inc.   
6          Paramount+          Paramount+         Paramount Global   
7             Peacock             Peacock                  Comcast   
8  Amazon Prime Video  Amazon Prime Video          Amazon.com Inc.   
9             Shudder                 NaN                      NaN   

                                    content_focus  \
0                          Original series, films   
1                                             NaN   
2  Family, Marvel, Star Wars, National Geographic   
3                

In [33]:
PlatformMerge_df['_merge'].value_counts()

_merge
both          8
left_only     2
right_only    0
Name: count, dtype: int64

In [34]:
AddPlatforms_df = PlatformMerge_df[PlatformMerge_df['_merge'] == 'left_only'][['platform_name']]
print('Platforms to add to dimension table:')
print(AddPlatforms_df)

Platforms to add to dimension table:
  platform_name
1   Crunchyroll
9       Shudder


In [35]:
print('Records in AddPlatforms_df:', AddPlatforms_df.shape[0])

Records in AddPlatforms_df: 2


In [36]:
AddPlatformsDistinct_df = AddPlatforms_df.drop_duplicates()
print('Rows of original:', AddPlatforms_df.shape[0])
print('Rows of distinct df:', AddPlatformsDistinct_df.shape[0])
print(f'\nRecords removed: {AddPlatforms_df.shape[0] - AddPlatformsDistinct_df.shape[0]}')

Rows of original: 2
Rows of distinct df: 2

Records removed: 0


In [37]:
# Build full rows for the new platforms with known metadata
NewPlatformRows_df = pd.DataFrame([
    {'platform': 'Crunchyroll', 'parent_company': 'Sony Group Corporation',
     'content_focus': 'Anime, manga',     'business_model': 'Subscription',
     'launch_year': 2009, 'is_digital': True, 'media_sector': 'Streaming',
     'platform_age_years': 2026 - 2009, 'launch_decade': '2000s'},
    {'platform': 'Shudder',     'parent_company': 'AMC Networks',
     'content_focus': 'Horror, thriller', 'business_model': 'Subscription',
     'launch_year': 2015, 'is_digital': True, 'media_sector': 'Streaming',
     'platform_age_years': 2026 - 2015, 'launch_decade': '2010s'},
])
print('New platform rows:')
print(NewPlatformRows_df)

New platform rows:
      platform          parent_company     content_focus business_model  \
0  Crunchyroll  Sony Group Corporation      Anime, manga   Subscription   
1      Shudder            AMC Networks  Horror, thriller   Subscription   

   launch_year  is_digital media_sector  platform_age_years launch_decade  
0         2009        True    Streaming                  17         2000s  
1         2015        True    Streaming                  11         2010s  


In [38]:
DimPlatform = pd.concat([PlatformSummaryDistinct_df, NewPlatformRows_df], ignore_index=True)

In [39]:
print('Rows of original:', PlatformSummaryDistinct_df.shape[0])
print('Rows after concat:', DimPlatform.shape[0])
print(f'\nRecords added: {DimPlatform.shape[0] - PlatformSummaryDistinct_df.shape[0]}')

Rows of original: 12
Rows after concat: 14

Records added: 2


In [40]:
# SCD Type 2 attributes – initial load: all records are current
DimPlatform['effective_date'] = pd.to_datetime(DimPlatform['launch_year'].astype(str) + '-01-01')
DimPlatform['end_date']       = None
DimPlatform['is_current']     = True

In [41]:
DimPlatform['platform_key'] = DimPlatform.reset_index().index + 1

print('\nDimPlatform with surrogate key:')
display(DimPlatform[['platform_key','platform','parent_company','media_sector',
                      'business_model','launch_year','launch_decade','is_digital',
                      'platform_age_years','effective_date','end_date','is_current']])


DimPlatform with surrogate key:


,platform_key,platform,parent_company,media_sector,business_model,launch_year,launch_decade,is_digital,platform_age_years,effective_date,end_date,is_current
0,1,Netflix,Netflix Inc.,Streaming,Subscription + Ads,2007,2000s,True,19,2007-01-01,None,True
1,2,Disney+,The Walt Disney Company,Streaming,Subscription + Ads,2019,2010s,True,7,2019-01-01,None,True
2,3,Hulu,The Walt Disney Company,Streaming,Subscription + Ads,2008,2000s,True,18,2008-01-01,None,True
3,4,Amazon Prime Video,Amazon.com Inc.,Streaming,Prime bundle + Subscription,2006,2000s,True,20,2006-01-01,None,True
4,5,Paramount+,Paramount Global,Streaming,Subscription + Ads,2021,2020s,True,5,2021-01-01,None,True
5,6,Peacock,Comcast,Streaming,Subscription + Ads,2020,2020s,True,6,2020-01-01,None,True
6,7,Apple TV+,Apple Inc.,Streaming,Subscription,2019,2010s,True,7,2019-01-01,None,True
7,8,Max,Warner Bros. Discovery,Streaming,Subscription + Ads,2020,2020s,True,6,2020-01-01,None,True
8,9,Roku,Roku Inc.,Streaming,AVOD + Transactional,2008,2000s,True,18,2008-01-01,None,True
9,10,YouTube TV,Google,Streaming,AVOD + Subscription,2017,2010s,True,9,2017-01-01,None,True


In [42]:
print('Rows and columns of DimPlatform:', DimPlatform.shape)
print('Record Count of DimPlatform:', len(DimPlatform))
print('DimPlatform counts:', DimPlatform[['platform_key','platform','is_current']].count())

Rows and columns of DimPlatform: (14, 13)
Record Count of DimPlatform: 14
DimPlatform counts: platform_key    14
platform        14
is_current      14
dtype: int64


#### DIM_SUBSCRIPTION_PLAN (SCD Type 2)

In [45]:
PlanMerge_df = pd.merge(
    StreamingServiceDistinct_df[['service', 'price']],
    PlanDistinct_df,
    on=['service', 'price'],
    how='left',
    indicator=True
)
print(PlanMerge_df)

       service  price _merge
0    Apple TV+   4.99   both
1    Apple TV+   4.99   both
2    Apple TV+   4.99   both
3    Apple TV+   4.99   both
4    Apple TV+   4.99   both
..         ...    ...    ...
772    Shudder   6.99   both
773    Shudder   6.99   both
774    Shudder   6.99   both
775    Shudder   6.99   both
776    Shudder   6.99   both

[777 rows x 3 columns]


In [89]:
# # Distinct service + price combos — the dimension candidates
# PlanDistinct_df = StreamingServiceDistinct_df[['service', 'price']].drop_duplicates()

In [91]:
# print('Rows and columns of PlanDistinct_df:', PlanDistinct_df.shape)
# print('Record Count of PlanDistinct_df:', len(PlanDistinct_df))
# print('PlanDistinct_df counts:', PlanDistinct_df.count())

In [46]:
PlanMerge_df.head()

,service,price,_merge
0,Apple TV+,4.99,both
1,Apple TV+,4.99,both
2,Apple TV+,4.99,both
3,Apple TV+,4.99,both
4,Apple TV+,4.99,both


In [47]:
PlanMerge_df['_merge'].value_counts()

_merge
both          777
left_only       0
right_only      0
Name: count, dtype: int64

In [48]:
AddPlans_df = PlanMerge_df.filter(['service', 'price', '_merge']).query("_merge == 'left_only'")
print('Plans to add to dimension table:')
print(AddPlans_df)

Plans to add to dimension table:
Empty DataFrame
Columns: [service, price, _merge]
Index: []


In [49]:
print('Records in AddPlans_df:', AddPlans_df.shape[0])

Records in AddPlans_df: 0


In [50]:
AddPlansDistinct_df = AddPlans_df.drop_duplicates()
print('Rows of original:', AddPlans_df.shape[0])
print('Rows of distinct df:', AddPlansDistinct_df.shape[0])
print(f'\nRecords removed: {AddPlans_df.shape[0] - AddPlansDistinct_df.shape[0]}')

Rows of original: 0
Rows of distinct df: 0

Records removed: 0


In [51]:
print('Rows and columns of AddPlansDistinct_df:', AddPlansDistinct_df.shape)
print('Record Count of AddPlansDistinct_df:', len(AddPlansDistinct_df))
print('AddPlansDistinct_df counts:', AddPlansDistinct_df.count())

Rows and columns of AddPlansDistinct_df: (0, 3)
Record Count of AddPlansDistinct_df: 0
AddPlansDistinct_df counts: service    0
price      0
_merge     0
dtype: int64


All source pricing rows map to a plan candidate — no new records to add. Now building SCD Type 2 attributes: collapsing consecutive months at the same price into a single record with `effective_date`, `end_date`, and `is_current`.

In [52]:
plan_src = StreamingServiceDistinct_df.sort_values(['service', 'date']).copy()
plan_src['price_changed'] = plan_src.groupby('service')['price'].transform(
    lambda x: x != x.shift()
).fillna(True)
plan_src['plan_group'] = plan_src.groupby('service')['price_changed'].cumsum()

print('Price change events per service:')
changes = plan_src[plan_src['price_changed'] & (plan_src['date'] > plan_src['date'].min())]
print(changes[['service', 'date', 'price']].to_string())

Price change events per service:
         service       date  price
553    Apple TV+ 2019-11-01   4.99
588    Apple TV+ 2022-10-01   6.99
603    Apple TV+ 2024-01-01   9.99
735  Crunchyroll 2020-08-01   7.99
151      Disney+ 2019-11-01   6.99
167      Disney+ 2021-03-01   7.99
188      Disney+ 2022-12-01  10.99
198      Disney+ 2023-10-01  13.99
202      HBO Max 2020-05-01  14.99
235      HBO Max 2023-02-01  15.99
247         Hulu 2015-10-01  11.99
319         Hulu 2021-10-01  12.99
333         Hulu 2022-12-01  14.99
346         Hulu 2024-01-01  17.99
90       Netflix 2019-01-01   8.99
126      Netflix 2022-01-01   9.99
147      Netflix 2023-10-01  15.49
347   Paramount+ 2014-10-01   9.99
451   Paramount+ 2023-06-01  11.99
604      Peacock 2020-07-01   9.99
641      Peacock 2023-08-01  11.99
459  Prime Video 2016-04-01   8.99
552  Prime Video 2024-01-01  11.99
647      Shudder 2016-10-01   5.99
729      Shudder 2023-08-01   6.99


In [53]:
DimSubscriptionPlan = plan_src.groupby(['service', 'plan_group', 'price', 'price_tier']).agg(
    effective_date=('date', 'min'),
    last_date     =('date', 'max')
).reset_index()

max_eff = DimSubscriptionPlan.groupby('service')['effective_date'].transform('max')
DimSubscriptionPlan['is_current'] = DimSubscriptionPlan['effective_date'] == max_eff
DimSubscriptionPlan['end_date']   = DimSubscriptionPlan.apply(
    lambda r: None if r['is_current'] else r['last_date'], axis=1
)

/var/folders/8y/_nxp_0_n50q0qqq5tr00vwsr0000gn/T/ipykernel_12705/3136905119.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  DimSubscriptionPlan = plan_src.groupby(['service', 'plan_group', 'price', 'price_tier']).agg(


In [54]:
DimSubscriptionPlan['plan_key'] = DimSubscriptionPlan.reset_index().index + 1

print('\nDimSubscriptionPlan with surrogate key:')
display(DimSubscriptionPlan[['plan_key', 'service', 'price', 'price_tier',
                             'effective_date', 'end_date', 'is_current']])


DimSubscriptionPlan with surrogate key:


,plan_key,service,price,price_tier,effective_date,end_date,is_current
0,1,Apple TV+,4.99,Budget ($0-$7),2019-11-01,2022-09-01,False
1,2,Apple TV+,4.99,Mid ($7-$14),NaT,NaT,False
2,3,Apple TV+,4.99,Premium (>$14),NaT,NaT,False
3,4,Apple TV+,5.99,Budget ($0-$7),NaT,NaT,False
4,5,Apple TV+,5.99,Mid ($7-$14),NaT,NaT,False
...,...,...,...,...,...,...,...
1675,1676,Shudder,15.99,Mid ($7-$14),NaT,NaT,False
1676,1677,Shudder,15.99,Premium (>$14),NaT,NaT,False
1677,1678,Shudder,17.99,Budget ($0-$7),NaT,NaT,False
1678,1679,Shudder,17.99,Mid ($7-$14),NaT,NaT,False


In [55]:
print('Rows and columns of DimSubscriptionPlan:', DimSubscriptionPlan.shape)
print('Record Count of DimSubscriptionPlan:', len(DimSubscriptionPlan))
print('DimSubscriptionPlan counts:', DimSubscriptionPlan[['plan_key', 'service', 'price', 'is_current']].count())

Rows and columns of DimSubscriptionPlan: (1680, 9)
Record Count of DimSubscriptionPlan: 1680
DimSubscriptionPlan counts: plan_key      1680
service       1680
price         1680
is_current    1680
dtype: int64


### FACT_SUBSCRIPTION_PRICING (Transactional)

In [56]:
fact_src = plan_src[['service', 'platform_name', 'date', 'year', 'price',
                      'price_tier', 'price_change_mom', 'cumulative_price_increase',
                      'plan_group']].copy()

print('Fact source shape:', fact_src.shape)
print('Record Count:', len(fact_src))

Fact source shape: (777, 9)
Record Count: 777


In [57]:
fact_src.head()

,service,platform_name,date,year,price,price_tier,price_change_mom,cumulative_price_increase,plan_group
553,Apple TV+,Apple TV+,2019-11-01,2019,4.99,Budget ($0-$7),NaN,0.0,1
554,Apple TV+,Apple TV+,2019-12-01,2019,4.99,Budget ($0-$7),0.0,0.0,1
555,Apple TV+,Apple TV+,2020-01-01,2020,4.99,Budget ($0-$7),0.0,0.0,1
556,Apple TV+,Apple TV+,2020-02-01,2020,4.99,Budget ($0-$7),0.0,0.0,1
557,Apple TV+,Apple TV+,2020-03-01,2020,4.99,Budget ($0-$7),0.0,0.0,1


In [58]:
fact_src = pd.merge(
    fact_src,
    DimSubscriptionPlan[['service', 'plan_group', 'plan_key']],
    on=['service', 'plan_group'],
    how='left',
    indicator=True
)
print('Fact → DIM_SUBSCRIPTION_PLAN merge:')
print(fact_src['_merge'].value_counts())
fact_src = fact_src.drop(columns='_merge')

Fact → DIM_SUBSCRIPTION_PLAN merge:
_merge
both          32634
left_only         0
right_only        0
Name: count, dtype: int64


In [59]:
fact_src = pd.merge(
    fact_src,
    DimPlatform[['platform', 'platform_key']],
    left_on='platform_name',
    right_on='platform',
    how='left',
    indicator=True
)
print('Fact → DIM_PLATFORM merge:')
print(fact_src['_merge'].value_counts())
fact_src = fact_src.drop(columns=['_merge', 'platform'])

Fact → DIM_PLATFORM merge:
_merge
both          32634
left_only         0
right_only        0
Name: count, dtype: int64


In [60]:
fact_src['date_key'] = fact_src['date'].dt.strftime('%Y%m%d').astype(int)

print('Unmatched rows (no platform_key):', fact_src['platform_key'].isna().sum())
print('Unmatched rows (no plan_key):', fact_src['plan_key'].isna().sum())

Unmatched rows (no platform_key): 0
Unmatched rows (no plan_key): 0


In [61]:
FactSubscriptionPricing = fact_src[['date_key', 'platform_key', 'plan_key',
    'price', 'price_change_mom', 'cumulative_price_increase']].copy()

print('\nFactSubscriptionPricing:')
display(FactSubscriptionPricing.head())


FactSubscriptionPricing:


,date_key,platform_key,plan_key,price,price_change_mom,cumulative_price_increase
0,20191101,7,1,4.99,NaN,0.0
1,20191101,7,2,4.99,NaN,0.0
2,20191101,7,3,4.99,NaN,0.0
3,20191101,7,4,4.99,NaN,0.0
4,20191101,7,5,4.99,NaN,0.0


In [62]:
print('Rows and columns of FactSubscriptionPricing:', FactSubscriptionPricing.shape)
print('Record Count of FactSubscriptionPricing:', len(FactSubscriptionPricing))
print('FactSubscriptionPricing counts:', FactSubscriptionPricing.count())

Rows and columns of FactSubscriptionPricing: (32634, 6)
Record Count of FactSubscriptionPricing: 32634
FactSubscriptionPricing counts: date_key                     32634
platform_key                 32634
plan_key                     32634
price                        32634
price_change_mom             32214
cumulative_price_increase    32634
dtype: int64
